# s2ISM: Multi-Plane ML-EM Reconstruction for Image Scanning Microscopy

This notebook walks through the full s2ISM pipeline:

1. **Data generation** — synthesise the tubulin phantom, find the optimal background plane, simulate PSFs, apply the noisy forward model
2. Load synthetic ISM data
3. Visualise the ground truth, measurements, and PSF
4. Load precomputed reconstruction results
5. Compare reconstruction quality against the raw ISM sum
6. Show axial sectioning

The notebook loads precomputed results from `evaluation/reference_outputs/` and runs in seconds. Heavy steps (PSF simulation, full reconstruction) are shown but kept commented out so the notebook stays interactive.

In [ ]:
import sys
sys.path.insert(0, '..')

import json
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

%matplotlib inline

task_dir = Path('..').resolve()
data_dir = task_dir / 'data'
ref_dir = task_dir / 'evaluation' / 'reference_outputs'

## 1. Data Generation

The dataset is fully synthetic. The generator (`src/generate_data.py`) is split into four functions, each handling one stage of the pipeline:

| Stage | Function | What it does |
|---|---|---|
| 1 | `make_tubulin_phantom` | Sample a 2-plane tubulin filament phantom in photon counts |
| 2 | `find_out_of_focus_from_param` | Compute the optimal axial distance $z^*$ for the second plane |
| 3 | `simulate_psfs` | Build a 25-channel SPAD PSF stack at $z^*$ |
| 4 | `apply_forward_model_with_noise` | Convolve, sum across planes, add Poisson noise |

### Why $z^*$ matters

The second plane of the PSF stack is **not** placed at an arbitrary spacing — it is placed at the depth that makes the in-focus and out-of-focus PSFs maximally distinguishable:

$$z^* = \arg\min_{z\,>\,0}\; \rho\!\bigl(h_0(\cdot),\, h_z(\cdot)\bigr)$$

where $\rho$ is the Pearson correlation between the PSF at the focal plane and at axial offset $z$. This is the **single most important assignment in data generation**: setting `gridPar.pxsizez` to anything other than $z^*$ produces two PSF planes that look almost identical, and the multi-plane reconstruction collapses to ordinary deconvolution.

In [ ]:
# The cached data already exists in `data/`. Inspect the imaging parameters:
with open(data_dir / 'meta_data.json') as f:
    meta = json.load(f)
print(json.dumps(meta, indent=2))

In [ ]:
# OPTIONAL — regenerate the data from scratch (~1 min). Uncomment to run.
#
# from src.generate_data import generate_data
# ground_truth, measurements_fresh, psf_fresh, meta_fresh = generate_data(
#     output_dir=str(data_dir), seed=42)
# print(f"Optimal background plane: {meta_fresh['optimal_bkg_plane_nm']} nm")

### Verify the optimal background plane assignment

We check that the saved PSF really was generated at $z^*$ by comparing the in-focus / out-of-focus plane Pearson correlation against a deliberately mis-spaced PSF (`pxsizez = 10 nm`). The correctly-spaced PSF should have noticeably *lower* correlation — that's the property $z^*$ minimises.

**This cell is heavy** (~30 s) because it builds two extra PSF stacks. Uncomment to run.

In [ ]:
# from scipy.stats import pearsonr
# from src.generate_data import simulate_psfs, make_psf_settings
#
# raw = np.load(data_dir / 'raw_data.npz')
# psf_saved = raw['psf'][0].astype(np.float64)
#
# exPar, emPar = make_psf_settings()
# pxsizex = meta['pxsizex_nm']
# Nz = meta['Nz']
# z_star = meta['optimal_bkg_plane_nm']
#
# psf_opt   = simulate_psfs(pxsizex, z_star, Nz, exPar, emPar, normalize=True)
# psf_wrong = simulate_psfs(pxsizex, 10.0,    Nz, exPar, emPar, normalize=True)
#
# def plane_corr(p):
#     return pearsonr(p[0].ravel(), p[1].ravel())[0]
#
# print(f'Saved PSF        plane-corr = {plane_corr(psf_saved):.4f}')
# print(f'Optimal pxsizez  plane-corr = {plane_corr(psf_opt):.4f}  (z = {z_star} nm)')
# print(f'Wrong   pxsizez  plane-corr = {plane_corr(psf_wrong):.4f}  (z = 10 nm)')

## 2. Load the cached data

In [ ]:
raw = np.load(data_dir / 'raw_data.npz')
gt_file = np.load(data_dir / 'ground_truth.npz')

measurements = raw['measurements'][0]      # (201, 201, 25)
psf = raw['psf'][0]                        # (2, 145, 145, 25)
ground_truth = gt_file['ground_truth'][0]  # (2, 201, 201)

print(f'Measurements: {measurements.shape}')
print(f'PSF:          {psf.shape}')
print(f'Ground truth: {ground_truth.shape}')

## 3. Visualise Ground Truth and Measurements

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

axes[0].imshow(ground_truth[0], cmap='magma')
axes[0].set_title('Ground Truth (In-Focus, z=0)')

axes[1].imshow(ground_truth[1], cmap='magma')
axes[1].set_title('Ground Truth (Out-of-Focus, z=1)')

ism_sum = measurements.sum(-1)
axes[2].imshow(ism_sum, cmap='magma')
axes[2].set_title('Raw ISM Sum (all 25 channels)')

plt.tight_layout()
plt.show()

## 4. Visualise PSF

The PSF has shape `(Nz, Ny, Nx, Nch)` — 2 z-planes, 25 detector channels.

In [ ]:
fig, axes = plt.subplots(5, 5, figsize=(12, 12))
for ch in range(25):
    ax = axes[ch // 5, ch % 5]
    ax.imshow(psf[0, :, :, ch], cmap='hot')
    ax.set_title(f'Ch {ch}', fontsize=8)
    ax.axis('off')
fig.suptitle('PSF (z=0, in-focus) — 25 SPAD channels', fontsize=14)
plt.tight_layout()
plt.show()

## 5. Load Precomputed Reconstruction

In [ ]:
ref = np.load(ref_dir / 'reconstruction.npz')
recon = ref['reconstruction'][0]  # (2, 201, 201)
print(f'Reconstruction shape: {recon.shape}')

## 6. Compare Results

In [ ]:
from src.visualization import plot_results, compute_metrics

fig = plot_results(ground_truth[0], ism_sum, recon[0])
plt.show()

metrics = compute_metrics(ground_truth[0], recon[0], ism_sum)
print(f'\nReconstruction vs Ground Truth:')
print(f"  NCC:   {metrics['ncc_recon']:.4f}")
print(f"  NRMSE: {metrics['nrmse_recon']:.4f}")
print(f'\nRaw ISM Sum vs Ground Truth:')
print(f"  NCC:   {metrics['ncc_ism']:.4f}")
print(f"  NRMSE: {metrics['nrmse_ism']:.4f}")

## 7. Axial Sectioning

s2ISM separates in-focus and out-of-focus planes, while the raw ISM sum mixes them together.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(recon[0], cmap='magma')
axes[0].set_title('s2ISM: In-Focus (z=0)')
axes[1].imshow(recon[1], cmap='magma')
axes[1].set_title('s2ISM: Out-of-Focus (z=1)')
plt.tight_layout()
plt.show()

## 8. Run Full Reconstruction from Scratch (Optional)

Uncomment the cell below to re-run the ML-EM reconstruction (~3 min on CPU).

In [ ]:
# from src.solvers import max_likelihood_reconstruction
#
# recon_fresh, counts, diff, n_iter = max_likelihood_reconstruction(
#     measurements.astype(np.float64),
#     psf.astype(np.float64),
#     stop='fixed',
#     max_iter=50,
#     rep_to_save='last'
# )
# print(f'Finished in {n_iter} iterations, shape: {recon_fresh.shape}')
#
# metrics_fresh = compute_metrics(ground_truth[0], recon_fresh[0], ism_sum)
# print(f"NCC: {metrics_fresh['ncc_recon']:.4f}, NRMSE: {metrics_fresh['nrmse_recon']:.4f}")

## Conclusion

The s2ISM pipeline has two critical pieces:

1. **Data generation** must place the second PSF plane at the optimal background distance $z^*$ (about 720 nm for these parameters), found by minimising the in-focus / out-of-focus PSF Pearson correlation. Skipping this step is the most common reproduction failure mode.
2. **Reconstruction** uses a multi-plane ML-EM update that jointly deconvolves all 25 SPAD channels and separates the two axial planes.

The result is a clear improvement over the raw ISM sum:

| Method | NCC | NRMSE |
|--------|-----|-------|
| s2ISM Reconstruction | ~0.90 | ~0.14 |
| Raw ISM Sum          | ~0.52 | ~0.47 |